# Stage 3 — Merge verified labels into the pool + new manifest
Reads the Label Studio export, converts to YOLO, appends to the pool
(idempotent, stable splits), and writes a new versioned manifest.

In [5]:
import sys; sys.path.insert(0,'/workspace/lib')
import os
import dataset_lib as D
from labelstudio_lib import ls_export_to_yolo

In [6]:
EXPORT = '/dataset/ls_export/export.json'
INCOMING = '/dataset/incoming'
DATASET_ROOT = '/dataset'
NEW_VERSION = 'v1.1'   # bump each time you add data
assert os.path.isfile(EXPORT), 'Export Label Studio to /dataset/ls_export/export.json first'

In [7]:
# 1) parse verified LS export -> YOLO label files in a temp dir
tmp_labels = '/dataset/ls_export/yolo_labels'
written = ls_export_to_yolo(EXPORT, INCOMING, tmp_labels)
print(f'verified labels for {len(written)} images')

verified labels for 62 images


In [8]:
# append each verified image+labels to the append-only pool
# skip images whose verified label is empty (blank frames add no crops)
added, dup, skipped_empty = 0, 0, 0
for fn in os.listdir(INCOMING):
    stem = os.path.splitext(fn)[0]
    lbl = os.path.join(tmp_labels, stem + '.txt')
    if not os.path.exists(lbl):
        continue  # image wasn't in the verified export
    lines = [l for l in open(lbl).read().splitlines() if l.strip()]
    if not lines:
        skipped_empty += 1
        continue  # submitted but empty (blank frame) -> not added
    iid, split, new = D.add_to_pool(DATASET_ROOT, os.path.join(INCOMING, fn), lines)
    added += int(new); dup += int(not new)
print(f'pool: {added} new, {dup} already present, {skipped_empty} empty skipped. '
      f'Total: {len(D.list_pool(DATASET_ROOT))}')

pool: 28 new, 4 already present, 30 empty skipped. Total: 48061


In [9]:
# 3) write the new versioned manifest
path, man = D.write_manifest(DATASET_ROOT, NEW_VERSION, note='added verified incoming batch')
print('manifest:', path)
print('images:', man['n_images'], '| splits:', man['split_counts'])
print('class counts:', man['class_counts'])

manifest: /dataset/manifests/dataset-v1.1.json
images: 48061 | splits: {'train': 38413, 'val': 4799, 'test': 4849}
class counts: {'ErinaceusEuropaeus': 2220, 'SciurusCarolinensis': 2159, 'SciurusVulgaris': 2005, 'CapreolusCapreolus': 2162, 'CervusElaphus': 1989, 'VulpesVulpes': 2464, 'MelesMeles': 1984, 'Person': 2525, 'CapraHircus': 1281, 'OryctolagusCuniculus': 2022, 'ColumbaPalumbus': 1785, 'PhasianusColchicus': 2029, 'OvisAries': 1894, 'PasserDomesticus': 1587, 'DamaDama': 2184, 'BosTaurus': 2305, 'FelisCatus': 2114, 'MartesMartes': 1701, 'CanisFamiliaris': 2007, 'CalibrationPole': 1178, 'AccipiterGentilis': 1682, 'ButeoButeo': 1693, 'CappercaillieCock': 1886, 'CappercaillieHen': 2022, 'NumeniusArquata': 1590, 'NumeniusArquataChick': 1411, 'EquusCaballus': 1697, 'CorvusCorone': 1548, 'SusScrofa': 2240, 'MuntiacusReevesi': 1999}


Pool updated and a new manifest written. Run **Stage 4** to retrain on it.
Optionally clear `/dataset/incoming/` now (images are safely copied into the pool).